# 03 — Category Model Evaluation

**Goal of this notebook**

1. Reload the Day-7 pipeline (`artifacts/category_model.joblib`) and re-evaluate it
   on the held-out **validation** and **test** splits.
2. Produce a confusion matrix and per-class precision / recall / F1 for both splits.
3. Inspect the top-N n-gram features the model relies on per class (LogReg coefficients).
4. Quantify **confidence calibration** so we can pick a sensible confidence threshold
   for the Day-15 `/classify` endpoint.
5. Write the first draft of `docs/results.md` from these numbers.

> Day 8 deliverable per `docs/PROJECT_PLAN.md`. Outputs are stripped from the saved
> notebook; `scripts/build_nb03.py` regenerates it deterministically.

## 0. Setup

In [ ]:
import json

import sys

import time

from pathlib import Path



import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

import seaborn as sns

from sklearn.metrics import (

    classification_report,

    confusion_matrix,

    precision_recall_fscore_support,

)



_here = Path.cwd().resolve()

REPO = next((p for p in (_here, *_here.parents) if (p / "src").is_dir()), _here)

if str(REPO) not in sys.path:

    sys.path.insert(0, str(REPO))

sys.path.insert(0, str(REPO / "src"))



from ticket_router.models.category_classifier import (

    CATEGORIES,

    CategoryClassifier,

)



sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams["figure.figsize"] = (8, 5)

print("Python:", sys.version.split()[0])

print("Repo root:", REPO)

print("Target categories:", list(CATEGORIES))

## 1. Reload model + splits

We always reload the artifact instead of refitting so this notebook is
fully reproducible from the saved Day-7 model.

In [ ]:
model_path = REPO / "artifacts" / "category_model.joblib"

assert model_path.exists(), f"Train first: python scripts/train_category.py  (missing {model_path})"



clf = CategoryClassifier.load(model_path)

print("Loaded pipeline:", clf.pipeline)

print("Classes:", clf.classes_())

In [ ]:
def _load(name: str) -> tuple[list[str], list[str]]:

    df = pd.read_csv(REPO / "data" / "processed" / f"{name}.csv")

    return df["text"].astype(str).tolist(), df["category"].astype(str).tolist()



val_texts,  val_labels  = _load("val")

test_texts, test_labels = _load("test")

print(f"val  rows: {len(val_labels):,}")

print(f"test rows: {len(test_labels):,}")

## 2. Confusion matrix + per-class metrics

In [ ]:
def report(y_true, y_pred) -> dict:

    labels = list(CATEGORIES)

    p, r, f, s = precision_recall_fscore_support(

        y_true, y_pred, labels=labels, zero_division=0

    )

    return {

        "labels": labels,

        "precision": p, "recall": r, "f1": f, "support": s,

        "report": classification_report(y_true, y_pred, labels=labels, zero_division=0),

        "confusion": confusion_matrix(y_true, y_pred, labels=labels),

    }



t0 = time.perf_counter()

val_pred  = clf.predict(val_texts)

test_pred = clf.predict(test_texts)

infer_seconds = time.perf_counter() - t0

print(f"Inferred val+test in {infer_seconds:.2f}s "

      f"({infer_seconds/(len(val_labels)+len(test_labels))*1000:.2f} ms/ticket)")

In [ ]:
val_r  = report(val_labels,  val_pred)

test_r = report(test_labels, test_pred)

print("VAL classification report\n" + val_r["report"])

print("TEST classification report\n" + test_r["report"])

In [ ]:
def plot_cm(cm: np.ndarray, labels: list[str], title: str) -> None:

    fig, ax = plt.subplots(figsize=(7, 6))

    sns.heatmap(

        cm, annot=True, fmt="d", cmap="Blues",

        xticklabels=labels, yticklabels=labels, cbar=False, ax=ax,

    )

    ax.set_xlabel("Predicted")

    ax.set_ylabel("True")

    ax.set_title(title)

    plt.xticks(rotation=20)

    plt.yticks(rotation=20)

    plt.tight_layout()

    plt.show()



plot_cm(val_r["confusion"],  list(CATEGORIES), "Validation confusion matrix")

plot_cm(test_r["confusion"], list(CATEGORIES), "Test confusion matrix")

In [ ]:
per_class = pd.DataFrame(

    {

        "val_P":  val_r["precision"],

        "val_R":  val_r["recall"],

        "val_F1": val_r["f1"],

        "val_n":  val_r["support"].astype(int),

        "test_P": test_r["precision"],

        "test_R": test_r["recall"],

        "test_F1":test_r["f1"],

        "test_n": test_r["support"].astype(int),

    },

    index=list(CATEGORIES),

)

per_class.round(3)

## 3. Top-N features per class

LogReg coefficients over a TF-IDF FeatureUnion give us a faithful
interpretability window. We read the coefficient matrix from the trained
pipeline, rank the union of word + char n-gram features, and show the
top 15 positive contributors for each class.

In [ ]:
logreg = clf.pipeline.named_steps["clf"]

feature_names = clf.feature_names()

coef = logreg.coef_                 # shape (n_classes, n_features)

classes = clf.classes_()

print("feature_names len:", len(feature_names))

print("coef shape:", coef.shape, "classes:", classes)

In [ ]:
TOP = 15

top_features: dict[str, list[tuple[str, float]]] = {}

for i, cls in enumerate(classes):

    idx = np.argsort(coef[i])[::-1][:TOP]

    top_features[cls] = [(feature_names[j], float(coef[i, j])) for j in idx]



for cls, feats in top_features.items():

    print(f"\n[{cls}]  top {TOP} features")

    for name, score in feats:

        print(f"  {score:+8.3f}  {name}")

In [ ]:
n_classes = len(classes)

fig, axes = plt.subplots(n_classes, 1, figsize=(8, 1.7 * n_classes), sharex=False)

for ax, cls in zip(axes, classes):

    feats = top_features[cls][:10][::-1]  # bottom-up for horizontal bar

    ax.barh([f for f, _ in feats], [s for _, s in feats], color="#3b82f6")

    ax.set_title(f"Top features -> {cls}", loc="left", fontsize=11)

    ax.tick_params(axis="y", labelsize=9)

plt.tight_layout()

plt.show()

## 4. Confidence calibration + threshold sweep

Confidence is the max probability returned by `predict_proba`. We want to
know:

* How concentrated is the score mass on the predicted class?
* At what threshold does `predicted_label == true_label` exceed 95%?
* What fraction of tickets fall below each threshold? (for fallback routing)

In [ ]:
def confidence_block(texts, labels) -> pd.DataFrame:

    proba = clf.predict_proba(texts)

    pred = proba.argmax(axis=1)

    conf = proba.max(axis=1)

    correct = np.array([classes[p] == t for p, t in zip(pred, labels)])

    df = pd.DataFrame({"conf": conf, "correct": correct})

    return df



val_conf  = confidence_block(val_texts,  val_labels)

test_conf = confidence_block(test_texts, test_labels)

val_conf.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, df, name in [(axes[0], val_conf, "val"), (axes[1], test_conf, "test")]:

    bins = np.linspace(0, 1, 21)

    ax.hist(df[df.correct]["conf"], bins=bins, alpha=0.7, label="correct", color="#10b981")

    ax.hist(df[~df.correct]["conf"], bins=bins, alpha=0.7, label="wrong",   color="#ef4444")

    ax.set_title(f"Confidence histogram — {name}")

    ax.set_xlabel("max(proba)")

    ax.set_ylabel("# tickets")

    ax.legend()

plt.tight_layout()

plt.show()

In [ ]:
thresholds = np.round(np.arange(0.30, 0.96, 0.05), 2)

rows = []

for t in thresholds:

    for df, name in [(val_conf, "val"), (test_conf, "test")]:

        covered = df["conf"] >= t

        n_covered = int(covered.sum())

        if n_covered == 0:

            continue

        acc = float(df.loc[covered, "correct"].mean())

        coverage = float(n_covered / len(df))

        rows.append({"threshold": t, "split": name, "coverage": coverage, "acc_at_or_above": acc, "n": n_covered})

thr_df = pd.DataFrame(rows).round(4)

thr_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for split, color in [("val", "#3b82f6"), ("test", "#f59e0b")]:

    sub = thr_df[thr_df["split"] == split]

    ax.plot(sub["threshold"], sub["acc_at_or_above"], marker="o", color=color, label=f"acc >= threshold ({split})")

    ax.plot(sub["threshold"], sub["coverage"],       marker="s", color=color, linestyle="--", alpha=0.5, label=f"coverage ({split})")

ax.set_xlabel("Confidence threshold")

ax.set_ylabel("Accuracy / Coverage")

ax.set_title("Threshold sweep: accuracy-on-covered vs coverage")

ax.axvline(0.60, color="grey", linestyle=":", linewidth=1)

ax.legend(loc="lower left", fontsize=9)

plt.tight_layout()

plt.show()

## 5. Where do the mistakes go?

Quick look at the most frequent (true -> predicted) mis-classifications on the
test split. These are the failure modes we want to attack in Day 18's edge-case work.

In [ ]:
pairs = pd.DataFrame({"true": test_labels, "pred": test_pred})

wrong = pairs[pairs.true != pairs.pred]

confusion_pairs = (

    wrong.groupby(["true", "pred"]).size()

    .sort_values(ascending=False)

    .head(10)

    .rename("count")

    .reset_index()

)

confusion_pairs

In [ ]:
test_proba = clf.predict_proba(test_texts)

test_conf  = test_proba.max(axis=1)

wrong_with_conf = wrong.copy()

wrong_with_conf["conf"] = test_conf[pairs.true != pairs.pred]

wrong_with_conf.sort_values("conf", ascending=False).head(10)

## 6. Per-ticket latency on test (proxy for API p99)

We measure the classifier's end-to-end latency on the test set. The full
`/classify` request will add cleaning + DB write overhead, but the model
is the dominant cost.

In [ ]:
samples = 200 if len(test_texts) > 200 else len(test_texts)

sample_texts = test_texts[:samples]

timings = []

for _ in range(3):

    t0 = time.perf_counter()

    clf.predict_proba(sample_texts)

    timings.append((time.perf_counter() - t0) / samples * 1000)

print(f"Avg ms/ticket over 3 runs of {samples} tickets: {np.mean(timings):.3f} ms (min {min(timings):.3f}, max {max(timings):.3f})")

## 7. Persist eval metrics + draft results.md

Two artifacts come out of this notebook:

* `artifacts/eval_metrics.json` — numeric summary for downstream scripts.
* `docs/results.md`            — human-readable first draft of the Day-8 results report.

In [ ]:
def _pack(y_true, y_pred, proba, name):

    labels = list(CATEGORIES)

    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    return {

        "accuracy": float((np.array(y_true) == np.array(y_pred)).mean()),

        "macro_f1": float(np.mean(f)),

        "weighted_f1": float(np.average(f, weights=s)),

        "per_class": {

            labels[i]: {"precision": float(p[i]), "recall": float(r[i]),

                        "f1": float(f[i]), "support": int(s[i])}

            for i in range(len(labels))

        },

        "confusion_matrix": cm.tolist(),

        "labels": labels,

        "n": int(len(y_true)),

        "name": name,

    }



val_proba = clf.predict_proba(val_texts)

test_proba = clf.predict_proba(test_texts)



EVAL_METRICS_PATH = REPO / "artifacts" / "eval_metrics.json"

RESULTS_MD = REPO / "docs" / "results.md"

EVAL_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)



eval_payload = {

    "val":  _pack(val_labels,  val_pred,  val_proba,  "val"),

    "test": _pack(test_labels, test_pred, test_proba, "test"),

    "latency_ms_per_ticket": float(np.mean(timings)),

    "top_features_per_class": {

        cls: [{"feature": f, "coef": s} for f, s in feats]

        for cls, feats in top_features.items()

    },

    "classes": clf.classes_(),

}



EVAL_METRICS_PATH.write_text(json.dumps(eval_payload, indent=2), encoding="utf-8")

print("Wrote", EVAL_METRICS_PATH)

In [ ]:
def _row(label, d):

    return f"| {label} | {d['precision']:.3f} | {d['recall']:.3f} | {d['f1']:.3f} | {d['support']} |"



val_d  = eval_payload["val"]

test_d = eval_payload["test"]

lines: list[str] = []

lines += [

    "# Model Evaluation Results",

    "",

    "> Draft generated on Day 8. Final numbers filled in after Day 19's full benchmark.",

    "",

    "## Headline numbers",

    "",

    f"- Validation accuracy: **{val_d['accuracy']:.3f}**  (macro F1 = {val_d['macro_f1']:.3f})",

    f"- Test accuracy:       **{test_d['accuracy']:.3f}**  (macro F1 = {test_d['macro_f1']:.3f})",

    f"- Avg model latency: **{eval_payload['latency_ms_per_ticket']:.3f} ms/ticket**",

    "",

    "Target thresholds (from `docs/PROJECT_PLAN.md`): accuracy >= 0.88, sentiment F1 >= 0.80, API p99 < 100 ms.",

    "",

    "## Per-class metrics (test split)",

    "",

    "| Category | Precision | Recall | F1 | Support |",

    "|---|---|---|---|---|",

]

for label in CATEGORIES:

    lines.append(_row(label, test_d["per_class"][label]))



lines += [

    "",

    "## Top features per class",

    "",

]

for cls, feats in eval_payload["top_features_per_class"].items():

    top_words = ", ".join(f"`{f['feature']}`" for f in feats[:8])

    lines.append(f"- **{cls}** — {top_words}")



lines += [

    "",

    "## Failure modes observed",

    "",

    "(See notebook section 5 for the live table. Day 18 will expand this with the edge-case suite.)",

    "",

]



RESULTS_MD.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("Wrote", RESULTS_MD)

print("\n---- preview ----")

print("\n".join(lines[:25]))

## 8. Day-8 sign-off checklist

* [x] Confusion matrix visualised for val and test.
* [x] Per-class precision / recall / F1 on val and test.
* [x] Top-15 n-gram features per class extracted and rendered.
* [x] Confidence calibration examined; threshold sweep exported.
* [x] Failure modes listed (top confused pairs).
* [x] Latency proxy recorded (target: stay under 100 ms end-to-end).
* [x] `docs/results.md` first draft written.

**Risks logged for later days:**

* If Day 19's full latency benchmark on 1000 tickets exceeds 80 ms end-to-end,
  swap `solver="liblinear"` for `solver="saga"` with a smaller `max_iter`.
* If `Technical Setup` recall stays < 0.65, expand the keyword table in
  `src/ticket_router/preprocessing/dataset.py` and consider seed edge cases
  from `data/synthetic/` (Day 18).